## Banking-Ticket-Intent-Classification-with-LoRA-PEFT

Fine-tuning Large Language Models with LoRA and PEFT for intent classification of banking support tickets. The project covers data preparation, parameter-efficient fine-tuning, evaluation, and inference for automated banking ticket routing and categorization.


### Install & Import Libraries

In [1]:
# Check the clean Colab environment
import sys

print("Python:", sys.version)

!pip uninstall -y torch torchvision torchaudio

Python: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


In [2]:
!pip install -q \
    torch==2.11.0 \
    torchvision==0.26.0 \
    torchaudio==2.11.0 \
    torchao>=0.16.0 \
    --index-url https://download.pytorch.org/whl/cu128

ERROR: Could not find a version that satisfies the requirement torchao (from versions: none)
ERROR: No matching distribution found for torchao


In [1]:
!pip install -q -U transformers datasets peft trl evaluate accelerate matplotlib scikit-learn
!pip install -q "pandas==2.2.3" "torch==2.11.0" "jedi>=0.16"


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [12 lines of output]
      + meson setup C:\Users\asus\AppData\Local\Temp\pip-install-txjqg7nz\pandas_74cfaf57f21d47eabfbd7f539a2d67ae C:\Users\asus\AppData\Local\Temp\pip-install-txjqg7nz\pandas_74cfaf57f21d47eabfbd7f539a2d67ae\.mesonpy-07ddewue\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\asus\AppData\Local\Temp\pip-install-txjqg7nz\pandas_74cfaf57f21d47eabfbd7f539a2d67ae\.mesonpy-07ddewue\build\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.1
      Source dir: C:\Users\asus\AppData\Local\Temp\pip-install-txjqg7nz\pandas_74cfaf57f21d47eabfbd7f539a2d67ae
      Build dir: C:\Users\asus\AppData\Local\Temp\pip-install-txjqg7nz\pandas_74cfaf57f21d47eabfbd7f539a2d

In [2]:
import random
import torch
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


c:\Users\asus\Desktop\Problem Solving Class\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## Load Dataset

### Task: Intent Classification

We use **[BANKING77](https://huggingface.co/datasets/PolyAI/banking77)**, a real-world dataset of customer service queries for a banking app, each labeled with one of 77 fine-grained intents.

77 intents is a lot for a short live-coding session, so we will **keep only 6 intents** that are easy to tell apart conceptually, and **subsample** a small number of examples per class. This keeps things:

- **small** — trains in a few minutes on a single GPU,
- **easy to understand** — 6 clear categories instead of 77,
- **low-preprocessing** — text in, label out, nothing else to clean.

| Split | Examples/class | Total |
|---|---|---|
| Train | 40 | 240 |
| Test | 10 | 60 |

### Why this dataset?

- It's real customer support text (not synthetic), so it's a realistic use case.
- Labels are short, single-token-ish strings, easy for a small model to learn to reproduce.
- Because it's a **narrow, specialized vocabulary** ("recipient", "top-up", "exchange rate"...), a *general-purpose* base model has no way of guessing the exact expected label format — this is exactly the situation where fine-tuning shines over prompting.


In [3]:
import pandas as pd
from datasets import Dataset, DatasetDict, ClassLabel

base_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"

train_df = pd.read_csv(base_url + "train.csv")
test_df = pd.read_csv(base_url + "test.csv")

# Six intents
selected_intents = [
    "card_arrival",
    "exchange_rate",
    "lost_or_stolen_card",
    "transfer_not_received_by_recipient",
    "balance_not_updated_after_bank_transfer",
    "activate_my_card",
]

# Keep only selected classes
train_df = train_df[train_df["category"].isin(selected_intents)].copy()
test_df = test_df[test_df["category"].isin(selected_intents)].copy()

# Map labels to 0-5
label_to_id = {
    label: i for i, label in enumerate(selected_intents)
}

id_to_label = {
    i: label for label, i in label_to_id.items()
}

train_df["label"] = train_df["category"].map(label_to_id)
test_df["label"] = test_df["category"].map(label_to_id)

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

# Make label a ClassLabel
class_label = ClassLabel(names=selected_intents)

train_dataset = train_dataset.cast_column("label", class_label)
test_dataset = test_dataset.cast_column("label", class_label)

selected_dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(selected_dataset)
print("\nLabels:")
for i, label in id_to_label.items():
    print(i, "->", label)

Casting the dataset: 100%|██████████| 240/240 [00:00<00:00, 68112.39 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 848
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 240
    })
})

Labels:
0 -> card_arrival
1 -> exchange_rate
2 -> lost_or_stolen_card
3 -> transfer_not_received_by_recipient
4 -> balance_not_updated_after_bank_transfer
5 -> activate_my_card


In [4]:
filtered_train = selected_dataset["train"]
filtered_test = selected_dataset["test"]

print(f"Filtered train size: {len(filtered_train)}")
print(f"Filtered test size: {len(filtered_test)}")

Filtered train size: 848
Filtered test size: 240


In [5]:
import random

EXAMPLES_PER_CLASS_TRAIN = 40
EXAMPLES_PER_CLASS_TEST = 10

def subsample(dataset, n_per_class):
    indices = []

    for label_id in range(len(selected_intents)):
        class_indices = [
            i for i, label in enumerate(dataset["label"])
            if label == label_id
        ]

        random.shuffle(class_indices)
        indices.extend(class_indices[:n_per_class])

    random.shuffle(indices)
    return dataset.select(indices)

train_dataset = subsample(filtered_train, EXAMPLES_PER_CLASS_TRAIN)
test_dataset = subsample(filtered_test, EXAMPLES_PER_CLASS_TEST)

print(f"Final train size: {len(train_dataset)}")
print(f"Final test size:  {len(test_dataset)}")

Final train size: 240
Final test size:  60


In [6]:
# Look at a few examples
for example in train_dataset.select(range(5)):
    intent = id_to_label[example["label"]]

    print(f"Text:   {example['text']}")
    print(f"Intent: {intent}")
    print("-" * 60)

Text:   What steps do I have to take to activate my card?
Intent: activate_my_card
------------------------------------------------------------
Text:   My card hasn't shown up yet.
Intent: card_arrival
------------------------------------------------------------
Text:   Hello I made a bank account transfer from the UK. The transfer was a couple hours ago, nothing has shown up yet.  Can you check to see if everything okay. Please.
Intent: balance_not_updated_after_bank_transfer
------------------------------------------------------------
Text:   Why have I not gotten my new card?
Intent: card_arrival
------------------------------------------------------------
Text:   Why haven't I gotten my new card?
Intent: card_arrival
------------------------------------------------------------
